# Validation and Export Notebook


In [1]:
import pandas as pd

from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [2]:
connection_string = quote_plus(
    "DRIVER=/opt/homebrew/lib/libmsodbcsql.18.dylib;"
    "SERVER=localhost,1433;"
    "DATABASE=SundorDW;"
    "UID=sa;"
    "PWD=yourStrong(!)Password;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

print("Connected")

Connected


In [3]:
from sqlalchemy import text

with engine.connect() as conn:

    result = conn.execute(
        text("SELECT DB_NAME()")
    )

    print(result.scalar())

SundorDW


In [4]:
OUTPUT_FOLDER = "../Data/Generated/"

In [ ]:
# LOAD ROUTE DIMENSION
dim_route = pd.read_csv(
    OUTPUT_FOLDER + "dim_route.csv"
)

print(dim_route.shape)

dim_route.head()

(26, 14)


,Route_ID,City,Country,Region,Distance_KM,Route_Tier,Seasonal_Route,Launch_Date,Base_Daily_Demand,Fare_Low_USD,Fare_High_USD,Aircraft_Type,Base_Daily_Flights,Competition_Sensitivity
0,1,Athens,Greece,Greece,1200,1,False,2020-01-01,180,140,260,B737-900ER,3,High
1,2,Rhodes,Greece,Greece,800,1,False,2020-01-01,160,130,240,B737-900ER,3,High
2,3,Heraklion,Greece,Greece,950,2,True,2020-01-01,130,130,240,B737-800,2,Medium
3,4,Kefalonia,Greece,Greece,1350,3,True,2020-01-01,70,130,250,B737-800,1,Medium
4,5,Sitia,Greece,Greece,1050,3,True,2023-04-01,60,130,240,B737-800,1,Low


In [6]:
# Insert into SQL Server
dim_route.to_sql(
    "dim_route",
    engine,
    if_exists="append",
    index=False
)

print("dim_route loaded")

ProgrammingError: (pyodbc.ProgrammingError) ('42000', '[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Error converting data type varchar to numeric. (8114) (SQLExecDirectW)')
[SQL: INSERT INTO dim_route ([Route_ID], [City], [Country], [Region], [Distance_KM], [Route_Tier], [Seasonal_Route], [Launch_Date], [Base_Daily_Demand], [Fare_Low_USD], [Fare_High_USD], [Aircraft_Type], [Base_Daily_Flights], [Competition_Sensitivity]) VALU ... 1045 characters truncated ...  ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: (1, 'Athens', 'Greece', 'Greece', 1200, 1, 0, '2020-01-01', 180, 140, 260, 'B737-900ER', 3, 'High', 2, 'Rhodes', 'Greece', 'Greece', 800, 1, 0, '2020-01-01', 160, 130, 240, 'B737-900ER', 3, 'High', 3, 'Heraklion', 'Greece', 'Greece', 950, 2, 1, '2020-01-01', 130, 130, 240, 'B737-800', 2, 'Medium', 4, 'Kefalonia', 'Greece', 'Greece', 1350, 3, 1, '2020-01-01' ... 264 parameters truncated ... 0, '2024-05

In [7]:
print(dim_route.dtypes)

Route_ID                    int64
City                       object
Country                    object
Region                     object
Distance_KM                 int64
Route_Tier                  int64
Seasonal_Route               bool
Launch_Date                object
Base_Daily_Demand           int64
Fare_Low_USD                int64
Fare_High_USD               int64
Aircraft_Type              object
Base_Daily_Flights          int64
Competition_Sensitivity    object
dtype: object


In [8]:
from sqlalchemy import text

with engine.connect() as conn:

    result = conn.execute(text("""
        SELECT
            COLUMN_NAME,
            DATA_TYPE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = 'dim_route'
        ORDER BY ORDINAL_POSITION
    """))

    for row in result:
        print(row)

('Route_ID', 'int')
('City', 'varchar')
('Country', 'varchar')
('Region', 'varchar')
('Distance_KM', 'int')
('Route_Tier', 'varchar')
('Seasonal_Route', 'bit')
('Launch_Date', 'date')
('Base_Daily_Demand', 'int')
('Fare_Low_USD', 'decimal')
('Fare_High_USD', 'decimal')
('Aircraft_Type', 'varchar')
('Base_Daily_Flights', 'int')
('Competition_Sensitivity', 'decimal')


In [9]:
dim_route["Competition_Sensitivity"].unique()

array(['High', 'Medium', 'Low'], dtype=object)